# 06 — Registro MLflow del experimento BERT

Sube a MLflow los artefactos, métricas y configuración del modelo BERT
fine-tuneado para clasificación de riesgo EU AI Act.

**Prerequisito:** haber ejecutado `04_entrenamiento.ipynb` o `train.py`

In [1]:
import os
import sys
import json
from pathlib import Path

# Fix conflicto OpenMP en Windows
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import mlflow
import mlflow.pyfunc
from dotenv import load_dotenv


def _find_repo_root(start: Path) -> Path:
    """Sube desde start hasta encontrar .git o pyproject.toml (raíz del repo)."""
    for p in [start, *start.parents]:
        if (p / '.git').exists() or (p / 'pyproject.toml').exists():
            return p
    return start


ROOT = _find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PIPELINE_DIR = ROOT / 'src/classifier/bert_pipeline'
MODEL_DIR    = PIPELINE_DIR / 'models'
BERT_PATH    = MODEL_DIR / 'bert_model'

# Cargar credenciales MLflow desde src/classifier/.env
env_path = ROOT / 'src/classifier/.env'
load_dotenv(env_path)
mlflow_uri = os.environ.get('MLFLOW_TRACKING_URI', '')

if not mlflow_uri:
    print('⚠ MLFLOW_TRACKING_URI no encontrado en .env — usando MLflow local')
else:
    mlflow.set_tracking_uri(mlflow_uri)
    print(f'MLflow URI: {mlflow_uri}')

if not BERT_PATH.exists():
    raise FileNotFoundError(
        f'Modelo BERT no encontrado en {BERT_PATH}.\n'
        'Ejecuta primero: python src/classifier/bert_pipeline/train.py'
    )
print('Modelo BERT encontrado OK')

MLflow URI: https://18.201.64.41/
Modelo BERT encontrado OK


## 1. Cargar métricas del entrenamiento

In [2]:
# Leer trainer_state.json del mejor checkpoint
checkpoints_dir = MODEL_DIR / 'checkpoints'

def _checkpoint_num(d: Path) -> int:
    try:
        return int(d.name.rsplit('-', 1)[-1])
    except ValueError:
        return 0

checkpoints = sorted(
    [d for d in checkpoints_dir.iterdir() if d.is_dir() and d.name.startswith('checkpoint')],
    key=_checkpoint_num,
)

if not checkpoints:
    raise FileNotFoundError('No hay checkpoints en models/checkpoints/')

last_checkpoint = checkpoints[-1]
trainer_state   = json.loads((last_checkpoint / 'trainer_state.json').read_text())
model_config    = json.loads((BERT_PATH / 'config.json').read_text())

# Extraer métricas por epoch
eval_logs   = [entry for entry in trainer_state['log_history'] if 'eval_f1' in entry]
best_metric = trainer_state.get('best_metric')
best_checkpoint = trainer_state.get('best_model_checkpoint', '')

print(f'Mejor F1 macro (val): {best_metric:.4f}')
print(f'Mejor checkpoint    : {best_checkpoint.split("/")[-1] if best_checkpoint else "N/A"}')
print()
print('Métricas por epoch:')
for entry in eval_logs:
    print(f"  Epoch {entry['epoch']:.0f}: F1={entry['eval_f1']:.4f}  Loss={entry['eval_loss']:.4f}")

Mejor F1 macro (val): 0.7289
Mejor checkpoint    : C:\Users\rammu\Documents\Asignaturas Keepcoding\Proyecto final\proyecto-final\src\classifier\bert_pipeline\models\checkpoints\checkpoint-800

Métricas por epoch:
  Epoch 1: F1=0.6314  Loss=0.8821
  Epoch 2: F1=0.6822  Loss=0.7462
  Epoch 3: F1=0.7228  Loss=0.7142
  Epoch 4: F1=0.7289  Loss=0.6999


## 2. Registrar experimento en MLflow

In [3]:
mlflow.set_experiment('bert_clasificador_riesgo_ia')

# Parámetros de entrenamiento (del config del modelo)
params = {
    'model_name'    : model_config.get('_name_or_path', 'dccuchile/bert-base-spanish-wwm-cased'),
    'num_labels'    : model_config.get('num_labels', 4),
    'max_length'    : 256,
    'epochs'        : len(eval_logs),
    'label_mapping' : str(model_config.get('id2label', {})),
    'architecture'  : 'BertForSequenceClassification',
    'dataset'       : 'dataset_augmented (sintético, data augmentation)',
}

# Métricas por epoch
metrics_final = {
    f'val_f1_epoch_{int(entry["epoch"])}': round(entry['eval_f1'], 4)
    for entry in eval_logs
}
metrics_final['val_f1_best']    = round(best_metric, 4)
metrics_final['val_loss_final'] = round(eval_logs[-1]['eval_loss'], 4)

with mlflow.start_run(run_name='bert-base-spanish-wwm-cased-v1') as run:
    mlflow.log_params(params)
    mlflow.log_metrics(metrics_final)

    # Artefactos del modelo
    mlflow.log_artifact(str(BERT_PATH / 'config.json'),           artifact_path='model')
    mlflow.log_artifact(str(BERT_PATH / 'tokenizer.json'),        artifact_path='model')
    mlflow.log_artifact(str(BERT_PATH / 'tokenizer_config.json'), artifact_path='model')

    # Artefactos visuales
    for png in ['training_curves.png', 'confusion_matrix_bert.png',
                'f1_per_class_bert.png', 'comparacion_modelos.png']:
        png_path = MODEL_DIR / png
        if png_path.exists():
            mlflow.log_artifact(str(png_path), artifact_path='plots')

    # Label encoder si existe
    le_path = MODEL_DIR / 'label_encoder.joblib'
    if le_path.exists():
        mlflow.log_artifact(str(le_path), artifact_path='artifacts')

    run_id = run.info.run_id
    print(f'Run registrado: {run_id}')
    print('Experimento  : bert_clasificador_riesgo_ia')
    print(f'F1 best val  : {best_metric:.4f}')

MlflowException: API request to https://18.201.64.41/api/2.0/mlflow/experiments/get-by-name failed with timeout exception HTTPSConnectionPool(host='18.201.64.41', port=443): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=bert_clasificador_riesgo_ia (Caused by ConnectTimeoutError(<HTTPSConnection(host='18.201.64.41', port=443) at 0x1847ffee300>, 'Connection to 18.201.64.41 timed out. (connect timeout=120)')). To increase the timeout, set the environment variable MLFLOW_HTTP_REQUEST_TIMEOUT (default: 120, type: int) to a larger value.

## 3. Añadir métricas de evaluación sobre dataset_sintetico_v2

Si ya se ejecutó `evaluar_sintetico_bert.py`, leemos el CSV de resultados.

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

results_csv = ROOT / 'src/classifier/resultados_sintetico_bert.csv'

if results_csv.exists():
    df_res = pd.read_csv(results_csv)
    validas = df_res[df_res['prediccion'] != 'ERROR']
    y_true = validas['etiqueta_real']
    y_pred = validas['prediccion']

    f1_sintetico  = f1_score(y_true, y_pred, average='macro')
    acc_sintetico = accuracy_score(y_true, y_pred)

    with mlflow.start_run(run_id=run_id):
        mlflow.log_metrics({
            'test_sintetico_v2_f1_macro' : round(f1_sintetico, 4),
            'test_sintetico_v2_accuracy' : round(acc_sintetico, 4),
            'test_sintetico_v2_n_samples': len(validas),
        })
        mlflow.log_artifact(str(results_csv), artifact_path='evaluation')
        png_sintetico = ROOT / 'src/classifier/matriz_confusion_sintetico_bert.png'
        if png_sintetico.exists():
            mlflow.log_artifact(str(png_sintetico), artifact_path='evaluation')

    print('Métricas dataset_sintetico_v2 registradas:')
    print(f'  F1 macro : {f1_sintetico:.4f}')
    print(f'  Accuracy : {acc_sintetico:.4f}')
    print(f'  Muestras : {len(validas)}')
else:
    print(f'⚠ No encontrado: {results_csv.name}')
    print('  Ejecuta evaluar_sintetico_bert.py primero para añadir estas métricas.')

## 4. Resumen

In [ ]:
print('=== REGISTRO MLFLOW COMPLETADO ===')
print(f'Run ID      : {run_id}')
print('Experimento : bert_clasificador_riesgo_ia')
print(f'URI MLflow  : {mlflow.get_tracking_uri()}')
print()
print('Artefactos registrados:')
print('  model/config.json')
print('  model/tokenizer.json')
print('  model/tokenizer_config.json')
print('  plots/*.png (curvas, matrices de confusión)')
print('  artifacts/label_encoder.joblib')
print('  evaluation/resultados_sintetico_bert.csv (si existe)')